In [1]:
from glob import glob

In [2]:
from os.path import join

In [3]:
import numpy as np

In [4]:
import torch

In [5]:
from safetensors_layer_grabber import yield_keys_and_tensors

In [6]:
safetensors_file_names = glob(join('Qwen', 'Qwen3-30B-A3B', '*.safetensors'))

------

In [7]:
# def qwen3_moe_rms_norm(
#     hidden_states,
#     weight,
# ):
#     input_dtype = hidden_states.dtype
#     hidden_states_float32 = hidden_states.to(torch.float32)
#     variance = hidden_states_float32.pow(2).mean(-1, keepdim=True)
#     hidden_states_float32_times_rsqrt_variance = hidden_states_float32 * torch.rsqrt(variance + 1e-06)
#     return weight * hidden_states_float32_times_rsqrt_variance.to(input_dtype)
def qwen3_moe_rms_norm(hidden_states, weight):
    # Save dtype
    input_dtype = hidden_states.dtype
    # Convert to float32 if needed
    hidden_states_float32 = hidden_states.astype(np.float32)
    # Calculate variance (mean of squared values along last axis)
    variance = np.mean(np.square(hidden_states_float32), axis=-1, keepdims=True)
    # Compute hidden_states * rsqrt(variance + 1e-6)
    hidden_states_float32_times_rsqrt_variance = hidden_states_float32 / np.sqrt(variance + 1e-6)
    # Multiply by weight (should be broadcastable)
    result = weight * hidden_states_float32_times_rsqrt_variance.astype(input_dtype)
    return result

In [8]:
inputs_and_outputs = torch.load('model.layers.32.input_layernorm.pt', weights_only=False)

In [9]:
hidden_states = inputs_and_outputs['hidden_states'].to(torch.float32).numpy()

In [10]:
hidden_states

array([[[-0.23925781,  0.27929688,  0.14941406, ..., -0.29882812,
          0.25195312, -0.10449219],
        [ 0.6953125 , -0.5703125 , -0.18359375, ...,  0.33203125,
          0.7421875 ,  0.08154297],
        [ 0.3984375 ,  0.09912109,  0.12402344, ..., -0.09570312,
         -0.06396484, -0.04101562],
        ...,
        [ 0.75      ,  0.93359375, -0.296875  , ...,  0.46484375,
          1.1328125 , -0.28125   ],
        [ 0.18164062,  0.13183594, -0.21191406, ..., -0.1953125 ,
         -0.03710938, -0.4453125 ],
        [-0.12207031, -0.27148438,  0.37109375, ...,  0.22265625,
          0.08300781,  0.05566406]]], shape=(1, 18, 2048), dtype=float32)

In [11]:
for tensor_key, tensor in yield_keys_and_tensors(safetensors_file_names):
    if tensor_key == 'model.layers.32.input_layernorm.weight':
        weight = tensor.to(torch.float32).numpy()

In [12]:
weight

array([0.14355469, 0.09667969, 0.11279297, ..., 0.12597656, 0.13574219,
       0.08349609], shape=(2048,), dtype=float32)

In [13]:
qwen3_moe_rms_norm(hidden_states, weight)

array([[[-0.00122143,  0.00096025,  0.00059932, ..., -0.00133874,
          0.00121624, -0.00031027],
        [ 0.1292109 , -0.07137562, -0.0268066 , ...,  0.05414651,
          0.13041581,  0.00881362],
        [ 0.09044616,  0.01515353,  0.02212067, ..., -0.01906463,
         -0.01372994, -0.00541537],
        ...,
        [ 0.11821862,  0.0991061 , -0.0367674 , ...,  0.06429897,
          0.16884187, -0.02578493],
        [ 0.04173243,  0.02039916, -0.03825473, ..., -0.03937886,
         -0.00806198, -0.05950787],
        [-0.0411372 , -0.06161514,  0.09825914, ...,  0.06584639,
          0.02645094,  0.01091059]]], shape=(1, 18, 2048), dtype=float32)

In [14]:
inputs_and_outputs['return'].to(torch.float32).numpy()

array([[[-0.0012207 ,  0.0009613 ,  0.00059891, ..., -0.00133514,
          0.0012207 , -0.0003109 ],
        [ 0.12890625, -0.07128906, -0.0267334 , ...,  0.05419922,
          0.13085938,  0.00878906],
        [ 0.09033203,  0.01519775,  0.02209473, ..., -0.01904297,
         -0.01373291, -0.00543213],
        ...,
        [ 0.11816406,  0.09912109, -0.03686523, ...,  0.06445312,
          0.16894531, -0.02575684],
        [ 0.04174805,  0.02038574, -0.03833008, ..., -0.03930664,
         -0.00805664, -0.05932617],
        [-0.04125977, -0.06152344,  0.09814453, ...,  0.06591797,
          0.02648926,  0.01092529]]], shape=(1, 18, 2048), dtype=float32)

------

In [15]:
# def silu_activation(input):
#     return torch.nn.functional.silu(input)
def silu_activation(input):
    return input * (1 / (1 + np.exp(-input)))

In [16]:
inputs_and_outputs = torch.load('model.layers.19.mlp.experts.act_fn.pt', weights_only=False)

In [17]:
input = inputs_and_outputs['input'].to(torch.float32).numpy()

In [18]:
input

array([[-2.28125   , -0.32617188, -0.75390625, ..., -0.796875  ,
        -0.90625   , -0.3359375 ],
       [-3.0625    , -0.6484375 , -0.71484375, ..., -0.41796875,
        -0.16699219, -1.0078125 ],
       [-0.11132812, -0.83203125, -0.28320312, ..., -0.14648438,
        -0.3359375 , -0.546875  ],
       ...,
       [-0.37109375, -0.3125    , -0.8515625 , ..., -0.2890625 ,
        -1.6796875 ,  0.1328125 ],
       [-0.36914062, -0.01879883, -0.18359375, ..., -0.12402344,
        -0.28710938,  0.32617188],
       [-0.59375   ,  0.71875   ,  1.3671875 , ..., -0.8125    ,
        -0.17480469, -2.0625    ]], shape=(144, 768), dtype=float32)

In [19]:
silu_activation(input)

array([[-0.21144399, -0.13672224, -0.24122795, ..., -0.2475846 ,
        -0.2607896 , -0.14001763],
       [-0.13683517, -0.22263566, -0.2348472 , ..., -0.1659348 ,
        -0.07654065, -0.26949725],
       [-0.05256877, -0.25228497, -0.12168351, ..., -0.06788734,
        -0.14001763, -0.20047878],
       ...,
       [-0.15150896, -0.13203269, -0.25470674, ..., -0.12378622,
        -0.26394078,  0.07080957],
       [-0.15088575, -0.00931107, -0.0833938 , ..., -0.05817119,
        -0.12308714,  0.18944964],
       [-0.21124135,  0.4832384 ,  1.0895464 , ..., -0.2497284 ,
        -0.07978257, -0.23264052]], shape=(144, 768), dtype=float32)

In [20]:
inputs_and_outputs['return'].to(torch.float32).numpy()

array([[-0.21191406, -0.13671875, -0.24121094, ..., -0.24804688,
        -0.26171875, -0.13964844],
       [-0.13671875, -0.22265625, -0.234375  , ..., -0.16601562,
        -0.07666016, -0.26953125],
       [-0.05249023, -0.25195312, -0.12158203, ..., -0.06787109,
        -0.13964844, -0.20019531],
       ...,
       [-0.15136719, -0.13183594, -0.25390625, ..., -0.12402344,
        -0.26367188,  0.07080078],
       [-0.15136719, -0.00933838, -0.08349609, ..., -0.05810547,
        -0.12304688,  0.18945312],
       [-0.2109375 ,  0.48242188,  1.0859375 , ..., -0.25      ,
        -0.07958984, -0.23242188]], shape=(144, 768), dtype=float32)

------

In [21]:
# def qwen3_moe_top_k_router(
#     hidden_states,
#     weight,
# ):
#     hidden_states_reshaped = hidden_states.reshape(-1, 2048)
#     router_logits = torch.nn.functional.linear(hidden_states_reshaped, weight)  # (seq_len, num_experts)
#     softmax_router_logits = torch.nn.functional.softmax(router_logits, dtype=torch.float, dim=-1)
#     router_top_value, router_indices = torch.topk(softmax_router_logits, 8, dim=-1)  # (seq_len, 8)
#     router_top_value_normed = router_top_value / router_top_value.sum(dim=-1, keepdim=True)
#     router_scores = router_top_value_normed.to(softmax_router_logits.dtype)
#     return softmax_router_logits, router_scores, router_indices
def qwen3_moe_top_k_router(
    hidden_states,  # (seq_length, 2048)
    weight  # (num_experts, 2048)
):
    # Reshape hidden_states to (-1, 2048)
    hidden_states_reshaped = hidden_states.reshape(-1, 2048)
    
    # Linear transformation (like torch.nn.functional.linear)
    # PyTorch's linear does: x @ W.T (so Weight is (num_experts, 2048))
    router_logits = np.dot(hidden_states_reshaped, weight.T)  # (seq_len, num_experts)
    
    # Softmax along last dimension
    exp_logits = np.exp(router_logits - np.max(router_logits, axis=-1, keepdims=True))
    softmax_router_logits = exp_logits / exp_logits.sum(axis=-1, keepdims=True)

    # Top-k (k=8) indices
    softmax_router_logits_argpartition = np.argpartition(softmax_router_logits, 8, axis=-1)
    top_8_softmax_router_logits_indices = softmax_router_logits_argpartition[:, -1:-9:-1]
    
    # Top-k (k=8) values
    router_top_values = np.take_along_axis(softmax_router_logits, top_8_softmax_router_logits_indices, axis=-1)
    router_scores = (router_top_values / router_top_values.sum(axis=-1, keepdims=True)).astype(softmax_router_logits.dtype)
    
    return softmax_router_logits, router_scores, top_8_softmax_router_logits_indices

In [22]:
inputs_and_outputs = torch.load('model.layers.30.mlp.gate.pt', weights_only=False)

In [23]:
hidden_states = inputs_and_outputs['hidden_states'].to(torch.float32).numpy()

In [24]:
hidden_states

array([[-0.00613403,  0.0065918 ,  0.00891113, ..., -0.01544189,
         0.01269531, -0.00248718],
       [ 0.265625  , -0.546875  ,  0.04321289, ...,  0.08203125,
        -0.07128906,  0.03540039],
       [ 0.546875  ,  0.15332031,  0.30859375, ..., -0.13964844,
        -0.1796875 , -0.07373047],
       ...,
       [ 0.59375   ,  0.5625    ,  0.16015625, ...,  0.21777344,
         0.14941406, -0.26953125],
       [ 0.10546875,  0.24121094, -0.44726562, ..., -0.04370117,
         0.28320312, -0.34960938],
       [-0.04101562, -0.0859375 ,  1.03125   , ...,  0.07470703,
        -0.07910156,  0.4453125 ]], shape=(18, 2048), dtype=float32)

In [25]:
for tensor_key, tensor in yield_keys_and_tensors(safetensors_file_names):
    if tensor_key == 'model.layers.30.mlp.gate.weight':
        weight = tensor.to(torch.float32).numpy()

In [26]:
weight

array([[ 1.4526367e-02,  2.8076172e-03, -2.9602051e-03, ...,
         2.2583008e-03, -5.7067871e-03,  1.7456055e-02],
       [-5.7220459e-05,  1.6967773e-02,  2.2583008e-03, ...,
        -2.2705078e-02,  2.3460388e-04,  1.1230469e-02],
       [ 6.9141388e-05,  4.3945312e-02,  1.7456055e-02, ...,
         3.6621094e-03, -1.5747070e-02,  9.0942383e-03],
       ...,
       [ 5.4931641e-03,  3.2196045e-03, -1.2329102e-02, ...,
         2.9373169e-04,  1.3977051e-02,  3.4484863e-03],
       [ 2.8686523e-02,  2.0996094e-02, -1.6113281e-02, ...,
         7.8125000e-03, -1.4221191e-02, -6.7749023e-03],
       [ 2.3651123e-03,  2.8076172e-02, -3.3721924e-03, ...,
        -1.7700195e-03, -7.1411133e-03, -6.3781738e-03]],
      shape=(128, 2048), dtype=float32)

In [27]:
qwen3_moe_top_k_router(hidden_states, weight)

(array([[2.1123784e-03, 1.2837113e-03, 8.5865526e-04, ..., 2.9395460e-03,
         1.2970025e-03, 2.3693338e-03],
        [7.1539534e-03, 4.8991716e-03, 6.2296860e-04, ..., 1.0994878e-02,
         9.4177583e-03, 1.6722431e-02],
        [6.0080085e-03, 2.2309071e-03, 8.7136636e-04, ..., 6.2793051e-03,
         8.4129469e-03, 6.1942055e-03],
        ...,
        [8.1242165e-03, 2.3121533e-03, 2.3744485e-04, ..., 5.2569965e-03,
         5.9340927e-03, 8.0651036e-03],
        [2.8156545e-03, 1.1349035e-03, 8.1131875e-05, ..., 3.4147063e-03,
         2.3127194e-03, 2.0200783e-02],
        [1.7885197e-03, 9.9871028e-04, 9.0157402e-05, ..., 1.8886371e-03,
         2.9543622e-03, 5.7499791e-03]], shape=(18, 128), dtype=float32),
 array([[0.9689683 , 0.00457169, 0.00455916, 0.00451087, 0.00438363,
         0.00437034, 0.00432702, 0.00430909],
        [0.20289795, 0.17450134, 0.13688254, 0.10206264, 0.10004853,
         0.09749718, 0.09329341, 0.09281639],
        [0.3035854 , 0.1515752 , 0.1214

In [28]:
inputs_and_outputs['return']

(tensor([[2.1067e-03, 1.2778e-03, 8.5118e-04,  ..., 2.9709e-03, 1.2778e-03,
          2.3872e-03],
         [7.2049e-03, 4.9519e-03, 6.2956e-04,  ..., 1.0816e-02, 9.5449e-03,
          1.6752e-02],
         [6.0379e-03, 2.2212e-03, 8.6985e-04,  ..., 6.2296e-03, 8.5148e-03,
          6.2296e-03],
         ...,
         [8.0781e-03, 2.3144e-03, 2.4394e-04,  ..., 5.2156e-03, 5.9101e-03,
          8.0781e-03],
         [2.8280e-03, 1.1426e-03, 8.2771e-05,  ..., 3.4112e-03, 2.3445e-03,
          2.0253e-02],
         [1.7692e-03, 1.0081e-03, 9.0879e-05,  ..., 1.8833e-03, 2.9169e-03,
          5.8010e-03]]),
 tensor([[0.9689, 0.0046, 0.0046, 0.0046, 0.0044, 0.0044, 0.0043, 0.0043],
         [0.2030, 0.1737, 0.1374, 0.1021, 0.1005, 0.0974, 0.0930, 0.0930],
         [0.3046, 0.1508, 0.1212, 0.1086, 0.1053, 0.0735, 0.0712, 0.0649],
         [0.2092, 0.1330, 0.1211, 0.1173, 0.1085, 0.1068, 0.1068, 0.0973],
         [0.2073, 0.1381, 0.1238, 0.1145, 0.1145, 0.1043, 0.0995, 0.0979],
         [0.236

------

In [29]:
# def qwen3_moe_rotary_embedding(
#     x,
#     position_ids,
#     inv_freq,
# ):
#     inv_freq_expanded = inv_freq[None, :, None].float().expand(position_ids.shape[0], -1, 1).to(x.device)
#     position_ids_expanded = position_ids[:, None, :].float()

#     freqs = (inv_freq_expanded.float() @ position_ids_expanded.float()).transpose(1, 2)
#     emb = torch.cat((freqs, freqs), dim=-1)
#     cos = emb.cos() * 1.0
#     sin = emb.sin() * 1.0

#     return cos.to(dtype=x.dtype), sin.to(dtype=x.dtype)
def qwen3_moe_rotary_embedding(
    x,
    position_ids,
    inv_freq,
):
    batch, seqlen = position_ids.shape
    
    # Expand dims to match shapes, similar to torch code
    inv_freq_expanded = np.expand_dims(inv_freq, axis=(0, 2))  # shape: (1, 64, 1)
    inv_freq_expanded_repeated = np.repeat(inv_freq_expanded, batch, axis=0)  # (batch, 64, 1)
    
    position_ids_expanded = np.expand_dims(position_ids, 1)  # (batch, 1, seqlen)

    # Matrix multiplication and transpose
    freqs = np.matmul(inv_freq_expanded_repeated.astype(np.float32), position_ids_expanded.astype(np.float32))  # (batch, 64, seqlen)
    freqs_transposed = np.transpose(freqs, (0, 2, 1))  # (batch, seqlen, 64)

    emb = np.concatenate((freqs_transposed, freqs_transposed), axis=-1)  # (batch, seqlen, 128)
    cos = np.cos(emb) * 1.0
    sin = np.sin(emb) * 1.0

    # Optionally convert cos/sin dtype to that of x
    cos = cos.astype(x.dtype)
    sin = sin.astype(x.dtype)

    return cos, sin

In [30]:
inputs_and_outputs = torch.load('model.rotary_emb.pt', weights_only=False)

In [31]:
inputs_and_outputs.keys()

odict_keys(['x', 'position_ids', 'return'])

In [32]:
inv_freq = torch.Tensor(
    [
        1.0000e+00, 8.0584e-01, 6.4938e-01, 5.2330e-01, 4.2170e-01, 3.3982e-01, 2.7384e-01, 2.2067e-01,
        1.7783e-01, 1.4330e-01, 1.1548e-01, 9.3057e-02, 7.4989e-02, 6.0430e-02, 4.8697e-02, 3.9242e-02,
        3.1623e-02, 2.5483e-02, 2.0535e-02, 1.6548e-02, 1.3335e-02, 1.0746e-02, 8.6596e-03, 6.9783e-03,
        5.6234e-03, 4.5316e-03, 3.6517e-03, 2.9427e-03, 2.3714e-03, 1.9110e-03, 1.5399e-03, 1.2409e-03,
        1.0000e-03, 8.0584e-04, 6.4938e-04, 5.2330e-04, 4.2170e-04, 3.3982e-04, 2.7384e-04, 2.2067e-04,
        1.7783e-04, 1.4330e-04, 1.1548e-04, 9.3057e-05, 7.4989e-05, 6.0430e-05, 4.8697e-05, 3.9242e-05,
        3.1623e-05, 2.5483e-05, 2.0535e-05, 1.6548e-05, 1.3335e-05, 1.0746e-05, 8.6596e-06, 6.9783e-06,
        5.6234e-06, 4.5316e-06, 3.6517e-06, 2.9427e-06, 2.3714e-06, 1.9110e-06, 1.5399e-06, 1.2409e-06,
    ]
)

In [33]:
inputs_and_outputs['position_ids']

tensor([[ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17]])

In [34]:
inputs_and_outputs['position_ids'][:, None, :]

tensor([[[ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
          17]]])

In [35]:
qwen3_moe_rotary_embedding(
    inputs_and_outputs['x'].to(dtype=torch.float32).numpy(),
    inputs_and_outputs['position_ids'].numpy(),
    inv_freq.numpy()
)

(array([[[ 1.        ,  1.        ,  1.        , ...,  1.        ,
           1.        ,  1.        ],
         [ 0.5403023 ,  0.6925055 ,  0.79645884, ...,  1.        ,
           1.        ,  1.        ],
         [-0.4161468 , -0.04087232,  0.2686934 , ...,  1.        ,
           1.        ,  1.        ],
         ...,
         [-0.7596879 ,  0.8875623 , -0.9505101 , ...,  1.        ,
           1.        ,  1.        ],
         [-0.9576595 ,  0.94698787, -0.5691682 , ...,  1.        ,
           1.        ,  1.        ],
         [-0.27516335,  0.42402694,  0.04387181, ...,  1.        ,
           1.        ,  1.        ]]], shape=(1, 18, 128), dtype=float32),
 array([[[ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00, ...,
           0.0000000e+00,  0.0000000e+00,  0.0000000e+00],
         [ 8.4147102e-01,  7.2141266e-01,  6.0469276e-01, ...,
           1.9110000e-06,  1.5399000e-06,  1.2409000e-06],
         [ 9.0929741e-01,  9.9916440e-01,  9.6322578e-01, ...,
           3.8220

In [36]:
inputs_and_outputs['return']

(tensor([[[ 1.0000,  1.0000,  1.0000,  ...,  1.0000,  1.0000,  1.0000],
          [ 0.5391,  0.6914,  0.7969,  ...,  1.0000,  1.0000,  1.0000],
          [-0.4160, -0.0408,  0.2695,  ...,  1.0000,  1.0000,  1.0000],
          ...,
          [-0.7578,  0.8867, -0.9492,  ...,  1.0000,  1.0000,  1.0000],
          [-0.9570,  0.9453, -0.5703,  ...,  1.0000,  1.0000,  1.0000],
          [-0.2754,  0.4238,  0.0439,  ...,  1.0000,  1.0000,  1.0000]]],
        dtype=torch.bfloat16),
 tensor([[[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
            0.0000e+00,  0.0000e+00],
          [ 8.3984e-01,  7.2266e-01,  6.0547e-01,  ...,  1.9073e-06,
            1.5423e-06,  1.2442e-06],
          [ 9.1016e-01,  1.0000e+00,  9.6484e-01,  ...,  3.8147e-06,
            3.0845e-06,  2.4885e-06],
          ...,
          [ 6.4844e-01, -4.6094e-01, -3.1055e-01,  ...,  2.8610e-05,
            2.3127e-05,  1.8597e-05],
          [-2.8711e-01,  3.2227e-01, -8.2031e-01,  ...,  3.0518e-05,
        

------